In [ ]:
!pip install -q transformers accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("hf_token"))

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

model_id = "google/gemma-3-4b-it"

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto", torch_dtype=torch.bfloat16
).eval()

processor = AutoProcessor.from_pretrained(model_id)

print(f"Model loaded — dtype: {next(model.parameters()).dtype}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Model loaded — dtype: torch.bfloat16


In [ ]:
from PIL import Image
import requests
from io import BytesIO
import torch

# Download image from URL
url = "https://www.ilankelman.org/stopsigns/australia.jpg"
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this image."},
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

print(processor.decode(generation, skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Here's a description of the image:

**Overall Impression:**

The image captures a vibrant intersection in a city with a strong Chinese cultural influence. It’s a sunny day, and the scene is a blend of modern urban elements and traditional Chinese architecture.

**Key Elements:**

* **Stop Sign:** A prominent red stop sign dominates the foreground, immediately drawing the viewer's attention.
* **Chinese Gate:** A large, ornate red and gold Chinese gate is the focal point of the background. It’s adorned with traditional Chinese characters (likely a welcoming phrase) and intricate detailing.
* **Lion Statues:** Two white stone lion statues flank the entrance to the gate, adding to the cultural theme.
* **Buildings:** The buildings surrounding the intersection are a mix of modern commercial establishments (with signs for "DAT" and "Optus") and traditional Chinese shops with red facades.
* **Street Scene:** A car is driving through the intersection, and a


In [ ]:
baseline_stats = {}
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        w = module.weight.data.float()
        flat = w.reshape(-1, 32)
        scale = flat.abs().amax(dim=1, keepdim=True) / 7.0  # ← Changed from 127.0 to 7.0 for INT4
        fractional = (flat / scale - (flat / scale).round()).abs()
        baseline_stats[name] = {
            "unique_vals": flat[0].unique().numel(),
            "mean_fractional": fractional.mean().item(),
        }

print(f"Baseline stats captured for {len(baseline_stats)} Linear layers ✓")

Baseline stats captured for 401 Linear layers ✓


In [ ]:
import torch

def mx_quantize(tensor, bits, block_size):
    original_shape = tensor.shape
    flat = tensor.reshape(-1)

    pad = (block_size - flat.numel() % block_size) % block_size
    if pad > 0:
        flat = torch.cat([flat, torch.zeros(pad, device=flat.device, dtype=flat.dtype)])

    blocks = flat.reshape(-1, block_size)
    block_max = blocks.abs().amax(dim=1, keepdim=True)

    # MX spec-compliant: shared_exp = floor(log2(max)) - emax_elem
    max_val = 2 ** (bits - 1) - 1  # 7 for INT4
    emax_elem = int(torch.floor(torch.log2(torch.tensor(float(max_val)))))  # 2 for INT4
    shared_exp = torch.floor(torch.log2(block_max)) - emax_elem
    # E8M0 format: clamp exponent to [-127, 127] range
    shared_exp = shared_exp.clamp(-127, 127)
    scale = 2.0 ** shared_exp
    scale = scale.clamp(min=1e-8)

    quantized = (blocks / scale).round().clamp(-max_val, max_val)
    dequantized = quantized * scale

    return dequantized.reshape(-1)[:original_shape.numel()].reshape(original_shape)


# Apply MXINT4 block-32
count = 0
with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            module.weight.data = mx_quantize(module.weight.data, bits=4, block_size=32)
            count += 1

print(f"MXINT4 block-32 applied to {count} Linear layers")

MXINT4 block-32 applied to 401 Linear layers


In [ ]:
results = []
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        flat = module.weight.data.float().cpu().reshape(-1, 32)

        # Use POWER-OF-2 scales for verification (matching quantization)
        block_max = flat.abs().amax(dim=1, keepdim=True)
        emax_elem = 2  # ← Changed to 2 for INT4
        shared_exp = torch.floor(torch.log2(block_max)) - emax_elem
        scale = 2.0 ** shared_exp
        scale = scale.clamp(min=1e-8)

        fractional = (flat / scale - (flat / scale).round()).abs()
        unique_vals = flat[0].unique().numel()
        results.append((name, fractional.mean().item(), unique_vals))

print(f"Total Linear layers quantized: {len(results)}\n")

for idx in [0, len(results)//2, len(results)-1]:
    name, frac, uniq = results[idx]
    if baseline_stats:
        baseline_frac = baseline_stats[name]["mean_fractional"]
        baseline_uniq = baseline_stats[name]["unique_vals"]
        print(f"Layer: {name}")
        print(f"  Fractional (base → quant): {baseline_frac:.8f} → {frac:.8f}")
        print(f"  Unique vals (base → quant): {baseline_uniq} → {uniq}")
    else:
        print(f"Layer: {name}")
        print(f"  Fractional (quant): {frac:.8f}")
        print(f"  Unique vals (quant): {uniq}")

print(f"\nMax fractional across all layers: {max(f for _,f,_ in results):.8f}")
print(f"All layers ≤256 unique vals: {all(u <= 256 for _,_,u in results)}")

Total Linear layers quantized: 401

Layer: model.vision_tower.vision_model.encoder.layers.0.self_attn.k_proj
  Fractional (base → quant): 0.24162772 → 0.00000000
  Unique vals (base → quant): 32 → 7
Layer: model.language_model.layers.5.self_attn.o_proj
  Fractional (base → quant): 0.24222989 → 0.00000000
  Unique vals (base → quant): 32 → 9
Layer: lm_head
  Fractional (base → quant): 0.24208553 → 0.00000000
  Unique vals (base → quant): 32 → 10

Max fractional across all layers: 0.00005572
All layers ≤256 unique vals: True


In [ ]:
from PIL import Image
import torch

# Use the same downloaded image from earlier
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},  # ← Uses image object from earlier download
            {"type": "text", "text": "Describe this image."},
        ]
    }
]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generation = generation[0][input_len:]

print("MXINT4 Output:", processor.decode(generation, skip_special_tokens=True))

MXINT4 Output: Here's a description of the image:

**Overall Impression:** The image depicts a bustling street scene with a strong Asian cultural influence, likely in a city with a significant Chinese community. It has a vibrant, slightly chaotic feel with a mix of modern and traditional elements.

**Key Elements:**

* **Stop Sign:** A prominent red stop sign dominates the foreground, immediately drawing the viewer's attention. It's positioned on a metal pole, adding to the urban setting.
* **Lion Statues:** Two white stone lion statues stand flanking the street, adding a distinctly Chinese element and suggesting a cultural district.
* **Building Facades:** Behind the stop sign and lions are colorful, ornate buildings.  There are red and gold details, suggesting a traditional Chinese architecture style.  The buildings are flanked by red lanterns and signage.
* **Car:** A silver car is driving down the street, adding a sense of movement and activity.
* **Street Scene:** The


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import base64
from PIL import Image
import io
from tqdm import tqdm

DRIVE_FILE = "/content/drive/MyDrive/vqav2_5k.jsonl"

samples = []
with open(DRIVE_FILE, "r") as f:
    for line in tqdm(f, total=5000):
        entry = json.loads(line)
        img_bytes = base64.b64decode(entry["image_b64"])
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        samples.append({
            "question_id": entry["question_id"],
            "question": entry["question"],
            "image": img,
            "answers": entry["answers"],
        })

print(f"Loaded {len(samples)} samples ✓")

Mounted at /content/drive


100%|██████████| 5000/5000 [00:15<00:00, 316.02it/s]

Loaded 5000 samples ✓


In [ ]:

import time

batch = samples[:32]

batch_messages = [
    [{"role": "user", "content": [
        {"type": "image", "image": s["image"]},
        {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
    ]}]
    for s in batch
]

inputs = processor.apply_chat_template(
    batch_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    padding=True,
)
inputs = inputs.to(model.device)

start = time.time()
with torch.no_grad():
    print("Input shape:", inputs["input_ids"].shape)
    print("VRAM used (GB):", torch.cuda.memory_allocated() / 1e9)
    generated_ids = model.generate(**inputs, max_new_tokens=32)
elapsed = time.time() - start

input_len = inputs["input_ids"].shape[1]
preds = processor.batch_decode(
    generated_ids[:, input_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

sps = 32 / elapsed
print(f"32 samples in {elapsed:.1f}s — {sps:.2f} samples/sec — ETA for 5k: {5000/sps/60:.1f} mins\n")

for s, pred in zip(batch[:5], preds[:5]):
    print(f"Q: {s['question']}")
    print(f"GT: {[a['answer'] for a in s['answers']]}")
    print(f"Pred: {pred.strip().lower()}")
    print()

Input shape: torch.Size([32, 291])
VRAM used (GB): 11.60427776
32 samples in 9.5s — 3.38 samples/sec — ETA for 5k: 24.6 mins

Q: Where is he looking?
GT: ['down', 'down', 'at table', 'skateboard', 'down', 'table', 'down', 'down', 'down', 'down']
Pred: mid-air

Q: What are the people in the background doing?
GT: ['spectating', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching', 'watching']
Pred: watching.

Q: What is he on top of?
GT: ['table', 'table', 'table', 'picnic table', 'picnic table', 'picnic table', 'picnic table', 'picnic table', 'skateboard', 'picnic table']
Pred: a skateboard

Q: What website copyrighted the picture?
GT: ['foodiebakercom', 'foodiebakercom', 'foodiebaker', 'foodiebakercom', 'foodiebakercom', 'http://foodiebakercom', 'foodiebakercom', 'foodiebakercom', 'foodiebakercom', 'foodiebaker']
Pred: foodiebaker.com

Q: Is this a creamy soup?
GT: ['no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no']
Pred: no



In [ ]:
import json
import torch
import time

OUTPUT_FILE = "vqav2_mxint4_32_results.jsonl"
BATCH_SIZE = 32

with open(OUTPUT_FILE, "w") as f:
    pass
print("First run")

count = 0
start = time.time()

with open(OUTPUT_FILE, "a") as f_out:
    for i in range(0, len(samples), BATCH_SIZE):
        batch = samples[i:i+BATCH_SIZE]

        batch_messages = [
            [{"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
            ]}]
            for s in batch
        ]

        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=4)

        input_len = inputs["input_ids"].shape[1]
        preds = processor.batch_decode(
            generated_ids[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        for s, pred in zip(batch, preds):
            result = {
                "question_id": s["question_id"],
                "question": s["question"],
                "prediction": pred.strip().lower(),
                "gt_answers": s["answers"],
            }
            f_out.write(json.dumps(result) + "\n")
        f_out.flush()

        count += len(batch)
        if count % 1000 == 0:
            elapsed = time.time() - start
            sps = count / elapsed
            print(f"{count}/5000 — {sps:.2f} samples/sec — ETA: {(5000-count)/sps/60:.1f} mins")

print(f"\nDone — {count} samples written to {OUTPUT_FILE}")

First run
4000/5000 — 4.09 samples/sec — ETA: 4.1 mins
5000/5000 — 4.09 samples/sec — ETA: 0.0 mins

Done — 5000 samples written to vqav2_mxint4_32_results.jsonl


In [ ]:
import json
import re

def normalize(answer):
    # Lowercase
    answer = answer.lower()
    # Remove articles
    answer = re.sub(r'\b(a|an|the)\b', ' ', answer)
    # Handle comma between digits (100,978 → 100978)
    answer = re.sub(r'(\d),(\d)', r'\1\2', answer)
    # Replace punctuation except apostrophe and colon with space
    answer = re.sub(r"[^\w\s':]", ' ', answer)
    # Normalize whitespace
    answer = ' '.join(answer.split())
    return answer.strip()

total_score = 0.0
total = 0

with open("vqav2_mxint4_32_results.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        pred = normalize(entry["prediction"])
        gt_answers = [normalize(a["answer"]) for a in entry["gt_answers"]]

        # Count how many GT answers match prediction
        match_count = sum(1 for gt in gt_answers if gt == pred)

        # Official VQA score: min(match_count / 3, 1)
        score = min(match_count / 3, 1.0)
        total_score += score
        total += 1

accuracy = total_score / total * 100
print(f"Samples evaluated: {total}")
print(f"VQA Accuracy: {accuracy:.2f}%")

Samples evaluated: 5000
VQA Accuracy: 39.43%


In [ ]:
correct_examples = []
wrong_examples = []

all_entries = []
with open("vqav2_mxint4_32_results.jsonl", "r") as f:
    for line in f:
        all_entries.append(json.loads(line))

for entry in reversed(all_entries):
    pred = normalize(entry["prediction"])
    gt_answers = [normalize(a["answer"]) for a in entry["gt_answers"]]
    match_count = sum(1 for gt in gt_answers if gt == pred)
    score = min(match_count / 3, 1.0)

    if score == 1.0 and len(correct_examples) < 3:
        correct_examples.append((entry, pred, gt_answers, score))
    elif score == 0.0 and len(wrong_examples) < 3:
        wrong_examples.append((entry, pred, gt_answers, score))

    if len(correct_examples) == 3 and len(wrong_examples) == 3:
        break

print("=== CORRECT ===")
for entry, pred, gt_answers, score in correct_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

print("=== WRONG ===")
for entry, pred, gt_answers, score in wrong_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

=== CORRECT ===
Q: What is the weather like?
Pred: sunny
GT: ['clear', 'sunny', 'tranquil']

Q: Are their leaves on the tree?
Pred: yes
GT: ['yes']

Q: Is the ground cold?
Pred: no
GT: ['yes', 'no']

=== WRONG ===
Q: How many yellow stripes are painted on the street?
Pred: three
GT: ['20', '7', 'at least 10', '9', '1', '12', 'more than 10', '50', 'multiple']

Q: What color car in background?
Pred: black car
GT: ['red', 'black']

Q: How tall is the house?
Pred: medium
GT: ['2 story', '20 feet', '24 ft', '2 stories', 'not very tall']



In [ ]:
import json
import base64
from PIL import Image
import io
from tqdm import tqdm

DRIVE_FILE = "/content/drive/MyDrive/textvqa_5k.jsonl"

textvqa_samples = []
with open(DRIVE_FILE, "r") as f:
    for line in tqdm(f, total=5000):
        entry = json.loads(line)
        img_bytes = base64.b64decode(entry["image_b64"])
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        textvqa_samples.append({
            "question_id": entry["question_id"],
            "question": entry["question"],
            "image": img,
            "answers": entry["answers"],
        })

print(f"Loaded {len(textvqa_samples)} TextVQA samples ✓")

100%|██████████| 5000/5000 [00:36<00:00, 138.24it/s]

Loaded 5000 TextVQA samples ✓


In [ ]:
import time

batch = textvqa_samples[:32]

batch_messages = [
    [{"role": "user", "content": [
        {"type": "image", "image": s["image"]},
        {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
    ]}]
    for s in batch
]

inputs = processor.apply_chat_template(
    batch_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    padding=True,
)
inputs = inputs.to(model.device)

start = time.time()
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=32)
elapsed = time.time() - start

input_len = inputs["input_ids"].shape[1]
preds = processor.batch_decode(
    generated_ids[:, input_len:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

sps = 32 / elapsed
print(f"32 samples in {elapsed:.1f}s — {sps:.2f} samples/sec — ETA for 5k: {5000/sps/60:.1f} mins\n")

for s, pred in zip(batch[:5], preds[:5]):
    print(f"Q: {s['question']}")
    print(f"GT: {s['answers']}")
    print(f"Pred: {pred.strip().lower()}")
    print()

32 samples in 8.7s — 3.69 samples/sec — ETA for 5k: 22.6 mins

Q: what is the brand of this camera?
GT: ['nous les gosses', 'dakota', 'clos culombu', 'dakota digital', 'dakota', 'dakota', 'dakota digital', 'dakota digital', 'dakota', 'dakota']
Pred: dakota

Q: what does the small white text spell?
GT: ['copenhagen', 'copenhagen', 'copenhagen', 'copenhagen', 'copenhagen', 'thursday', 'copenhagen', 'copenhagen', 'copenhagen', 'copenhagen']
Pred: copenhagen

Q: what kind of beer is this?
GT: ['ale', 'sublimely self-righteous ale', 'stone', 'ale', 'self righteous', 'ale', 'ale', 'ale', 'ale', 'ale']
Pred: self-righteous ale

Q: what brand liquor is on the right?
GT: ['bowmore ', 'bowmore', 'bowmore', 'bowmore', 'bowmore', 'bowmore', 'bowmore', 'bowmore islay', 'dowmore islay', 'bowmore islay']
Pred: bowmore

Q: how long has the drink on the right been aged?
GT: ['10 years', '10 year', '10 years', '10 years ', '10 years', '10 years', '10 years', '10 years', 'martial arts', '10']
Pred: 10 ye

In [ ]:
import json
import torch
import time

OUTPUT_FILE = "textvqa_mxint4_32_results.jsonl"
BATCH_SIZE = 32

with open(OUTPUT_FILE, "w") as f:
    pass
print("First run")

count = 0
start = time.time()

with open(OUTPUT_FILE, "a") as f_out:
    for i in range(0, len(textvqa_samples), BATCH_SIZE):
        batch = textvqa_samples[i:i+BATCH_SIZE]

        batch_messages = [
            [{"role": "user", "content": [
                {"type": "image", "image": s["image"]},
                {"type": "text", "text": s["question"] + " Answer the question using a single word or phrase."},
            ]}]
            for s in batch
        ]

        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=32)

        input_len = inputs["input_ids"].shape[1]
        preds = processor.batch_decode(
            generated_ids[:, input_len:],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        for s, pred in zip(batch, preds):
            result = {
                "question_id": s["question_id"],
                "question": s["question"],
                "prediction": pred.strip().lower(),
                "gt_answers": s["answers"],
            }
            f_out.write(json.dumps(result) + "\n")
        f_out.flush()

        count += len(batch)
        elapsed = time.time() - start
        if (count // 1000) > ((count - len(batch)) // 1000):
            sps = count / elapsed
            print(f"{count}/5000 — {sps:.2f} samples/sec — ETA: {(5000-count)/sps/60:.1f} mins")

print(f"\nDone — {count} samples written to {OUTPUT_FILE}")

First run
1024/5000 — 3.59 samples/sec — ETA: 18.5 mins
2016/5000 — 3.63 samples/sec — ETA: 13.7 mins
3008/5000 — 3.61 samples/sec — ETA: 9.2 mins
4000/5000 — 3.61 samples/sec — ETA: 4.6 mins
5000/5000 — 3.62 samples/sec — ETA: 0.0 mins

Done — 5000 samples written to textvqa_mxint4_32_results.jsonl


In [ ]:
import json
import re

def normalize(answer):
    answer = answer.lower()
    answer = re.sub(r'\b(a|an|the)\b', ' ', answer)
    answer = re.sub(r'(\d),(\d)', r'\1\2', answer)
    answer = re.sub(r"[^\w\s':]", ' ', answer)
    answer = ' '.join(answer.split())
    return answer.strip()

total_score = 0.0
total = 0

with open("textvqa_mxint4_32_results.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        pred = normalize(entry["prediction"])
        gt_answers = [normalize(a) for a in entry["gt_answers"]]

        match_count = sum(1 for gt in gt_answers if gt == pred)
        score = min(match_count / 3, 1.0)
        total_score += score
        total += 1

accuracy = total_score / total * 100
print(f"Samples evaluated: {total}")
print(f"TextVQA Accuracy: {accuracy:.2f}%")

Samples evaluated: 5000
TextVQA Accuracy: 50.63%


In [ ]:
correct_examples = []
wrong_examples = []

all_entries = []
with open("textvqa_mxint4_32_results.jsonl", "r") as f:
    for line in f:
        all_entries.append(json.loads(line))

for entry in reversed(all_entries):
    pred = normalize(entry["prediction"])
    gt_answers = [normalize(a) for a in entry["gt_answers"]]
    match_count = sum(1 for gt in gt_answers if gt == pred)
    score = min(match_count / 3, 1.0)

    if score == 1.0 and len(correct_examples) < 3:
        correct_examples.append((entry, pred, gt_answers))
    elif score == 0.0 and len(wrong_examples) < 3:
        wrong_examples.append((entry, pred, gt_answers))

    if len(correct_examples) == 3 and len(wrong_examples) == 3:
        break

print("=== CORRECT ===")
for entry, pred, gt_answers in correct_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

print("=== WRONG ===")
for entry, pred, gt_answers in wrong_examples:
    print(f"Q: {entry['question']}")
    print(f"Pred: {pred}")
    print(f"GT: {list(set(gt_answers))}")
    print()

=== CORRECT ===
Q: what store is seen in the background?
Pred: pret manger
GT: ['pret manger']

Q: what number is on the black and white sign?
Pred: 201
GT: ['201', 'oobermind']

Q: what team name is on this man's jersey?
Pred: storm chasers
GT: ['storm chasers', 'chasers']

=== WRONG ===
Q: how many points does ga state have?
Pred: 81
GT: ['58']

Q: what is the highest number on the players shorts?
Pred: four
GT: ['8', '14']

Q: what brand is on the red and black jerseys?
Pred: reebo
GT: ['united emirates', 'erea', 'jan', 'unanswerable', 'nike']

